# 01 Data Splitting

## Purpose
This notebook creates the train, validation, and test splits for the rebuilt workflow.

## Why this notebook matters
The split created here becomes a shared foundation for the downstream preprocessing, training, and evaluation notebooks. Once the split is written to disk, later notebooks can reuse the same partition instead of creating their own.

## Inputs
- Raw glycan sequence file stored in the project folder on Google Drive

## Outputs
- `train.txt`
- `val.txt`
- `test.txt`
- `split_summary.csv`


## Runtime setup

This cell prepares the Colab environment for the notebook. It mounts Google Drive, synchronizes the GitHub repository, and makes the project `src` code available for import.

We keep the setup code explicit here because the notebook must first download the repository before it can import the shared helper modules.

**Expected output**
- confirmation that Google Drive is mounted
- confirmation that the GitHub repository is available locally
- confirmation of the active repository directory


In [ ]:
# Standard library imports used for environment setup.
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read project data and save outputs.
drive.mount('/content/drive')

# Define the public GitHub repository that stores the project notebooks and
# shared helper modules.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = Path('/content') / REPO_NAME

# Clone the repository into the Colab runtime the first time the notebook runs.
# If the repository is already present, pull the latest changes so the notebook
# uses the current helper code.
if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

print(f"Updating repository to the latest '{GITHUB_REF}' changes...")
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_REF],
    check=True,
)

# Add the repository root to the Python import path so the notebook can import
# shared helper modules from the src package.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

print(f'Repository directory: {REPO_DIR}')


## User settings

This is the main cell that should be reviewed before running the notebook.

Update `PROJECT_ROOT` so it points to the correct project folder in Google Drive. This cell also exposes the split-specific parameters that control how the reusable dataset partition is created.

**Settings to review**
- `PROJECT_ROOT`: the root folder for this project in Google Drive
- `RAW_DATA_FILENAME`: the raw glycan text file used in this notebook
- `OVERWRITE_EXISTING_OUTPUTS`: whether existing split files may be replaced
- `RANDOM_SEED`: the random seed used for the split; set it to `None` to generate and print a new seed
- `HELD_OUT_FRACTION`: the fraction of the full dataset reserved for validation plus test together

**Expected output**
- the resolved raw input path
- the split output directory
- the active split settings for this run


In [ ]:
from pathlib import Path

# Update PROJECT_ROOT if your Drive project folder has a different name or
# location. This is the main path value that should be checked before running.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# This notebook reads one raw text file that contains one glycan sequence per line.
RAW_DATA_FILENAME = 'raw_glycans_dataset_no_aldi.txt'

# If True, the notebook may replace previously saved split files in the target
# folder. If False, the notebook will stop before overwriting files.
OVERWRITE_EXISTING_OUTPUTS = False

# Provide an integer seed to reproduce a known split. Set this to None if you
# want the helper to generate a new seed and print it for later reuse.
RANDOM_SEED = 42

# This value controls the combined size of the validation and test sets. The
# helper splits this held-out portion evenly into validation and test subsets.
HELD_OUT_FRACTION = 0.20

# Build the input and output paths used by this notebook.
raw_data_path = PROJECT_ROOT / 'data' / 'raw' / RAW_DATA_FILENAME
splits_dir = PROJECT_ROOT / 'data' / 'splits'
splits_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data path: {raw_data_path}')
print(f'Split output directory: {splits_dir}')
print(f'Random seed setting: {RANDOM_SEED}')
print(f'Held-out fraction: {HELD_OUT_FRACTION}')


## Validate the required paths

This cell checks that the raw glycan dataset exists and that the planned output files can be written safely.

We do this early so that path problems and overwrite-policy problems are caught immediately instead of causing confusing downstream errors.

**Expected output**
- a confirmation that the input file exists
- a confirmation that the output paths are valid for this run

**How to interpret the result**
- if this cell raises a file error, `PROJECT_ROOT` or `RAW_DATA_FILENAME` likely needs to be corrected
- if this cell raises a file-exists error, the notebook found prior outputs and overwrite mode is disabled


In [ ]:
from src.notebook_utils import require_existing_path, validate_output_paths

# Verify that the notebook can find the raw input file before continuing.
require_existing_path(raw_data_path, 'Raw glycan dataset')

# Define the files this notebook expects to save. The shared helper enforces a
# consistent overwrite policy across notebooks.
output_paths = {
    'train_path': splits_dir / 'train.txt',
    'val_path': splits_dir / 'val.txt',
    'test_path': splits_dir / 'test.txt',
    'split_summary_path': splits_dir / 'split_summary.csv',
}

validate_output_paths(
    output_paths=output_paths,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
)

print('Input and output path checks passed.')


## Create the reusable dataset split

This cell runs the shared split helper in `src.data_utils.py`. The helper loads the raw sequences, resolves the active random seed, creates the train/validation/test split, writes the split files to disk, and returns a summary table.

**Expected output**
- the number of loaded sequences
- the resolved random seed used for this run
- counts for the train, validation, and test partitions
- the saved split text files and summary table

**How to interpret the result**
- if the seed was generated automatically, record the printed value if you want to reproduce the same split later
- the reported split counts should add up to the total number of usable sequences


In [ ]:
from src.data_utils import run_data_split_pipeline

# Run the shared split pipeline so the notebook stays focused on settings,
# interpretation, and quick verification rather than low-level file handling.
split_results = run_data_split_pipeline(
    raw_file_path=raw_data_path,
    output_dir=splits_dir,
    held_out_fraction=HELD_OUT_FRACTION,
    random_seed=RANDOM_SEED,
)

resolved_seed = split_results['random_seed']
split_summary_df = split_results['split_summary_df']

print(f'Resolved random seed: {resolved_seed}')


## Review the saved split summary

This cell displays the split summary table so the partition sizes can be checked directly in the notebook.

A saved copy of the same table is written to `split_summary.csv`, which makes it easy to confirm the split sizes later without reloading the text files.

**Expected output**
- a three-row summary table showing the number of sequences in each split

**How to interpret the result**
- the train split should be the largest partition
- the validation and test splits should be similar in size because the held-out set is split evenly


In [ ]:
from IPython.display import display

display(split_summary_df)
